# Broomstick Model - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC simulation results for a broomstick (cylinder) geometry using the ENDF/B-VIII.1 nuclear data library for neutron transport.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Tallies**: Flux, heating, and all available reaction MTs (energy-integrated and energy-binned)

In [1]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 5000000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # eV
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1  # Source energy + 10%
FLUX_DECADES = 6  # Number of decades to show on flux y-axis (from max value)

# Broomstick geometry: thin cylinder along z (radius 1 cm, height 100 cm).
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm (total height = 100 cm)

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Codes/backends to run. Comma-separated list via the MC_BACKENDS env var.
# Tokens:
#   openmc-cpu  - reference OpenMC simulation
#   yamc-cpu    - YAMC on CPU
#   yamc-gpu    - YAMC on GPU (requires a working f64 Vulkan adapter)
# Tokens are independent - any one, two, or all three are valid.
# Default runs all three. Examples:
#   MC_BACKENDS=yamc-cpu,yamc-gpu      # skip OpenMC (compare yamc backends)
#   MC_BACKENDS=openmc-cpu,yamc-cpu    # no GPU
_VALID_BACKENDS = ("openmc-cpu", "yamc-cpu", "yamc-gpu")
_raw = os.environ.get("MC_BACKENDS", "openmc-cpu,yamc-cpu,yamc-gpu")
MC_BACKENDS = [b.strip().lower() for b in _raw.split(",") if b.strip()]
_bad = [b for b in MC_BACKENDS if b not in _VALID_BACKENDS]
if _bad:
    raise ValueError(
        f"MC_BACKENDS contains unknown token(s) {_bad}; "
        f"valid tokens are {list(_VALID_BACKENDS)}"
    )
if not MC_BACKENDS:
    raise ValueError("MC_BACKENDS resolved to an empty list")
RUN_OPENMC = "openmc-cpu" in MC_BACKENDS
COMPUTES = [b.split("-", 1)[1] for b in MC_BACKENDS if b.startswith("yamc-")]
# No precheck for GPU availability - the dispatch raises a clean
# RuntimeError if no f64 Vulkan adapter is present, which is enough
# signal for both CI and interactive use.
print(f"Backends: {MC_BACKENDS}")


Backends: ['openmc-cpu', 'yamc-cpu', 'yamc-gpu']


In [2]:
# Results cache - load from disk if a previous run produced them with the
# same parameters, else run and write them out. OpenMC results are keyed by
# run parameters only (OpenMC is fixed). YAMC results additionally include
# the yamc version/commit, so a new yamc build auto-invalidates the cache
# and the simulation re-runs.
import sys
from pathlib import Path as _Path

_repo_root = _Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "cache_utils.py").exists():
    _repo_root = _repo_root.parent
if (_repo_root / "cache_utils.py").exists() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from cache_utils import cached_or_run, yamc_fingerprint

CACHE_DIR = _Path("cache")
OPENMC_CACHE_META = {
    "library": NUCLEAR_DATA,
    "particles": PARTICLES,
    "batches": BATCHES,
    "source_energy": SOURCE_ENERGY,
}
# Same run parameters plus the yamc version/commit, so the YAMC cache
# auto-invalidates whenever the installed yamc changes.
YAMC_CACHE_META = {**OPENMC_CACHE_META, **yamc_fingerprint()}

In [3]:
def run_openmc_simulation(nuclide, nuclear_data):
    """Run OpenMC simulation for a single nuclide."""
    import openmc as mc
    
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[nuclear_data]}cross_sections.xml"
    
    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(surface_id=2, r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = mc.ZPlane(surface_id=3, z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = mc.ZPlane(surface_id=4, z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region1 = -cylinder & +z_bot & -z_top
    
    # Material
    material1 = mc.Material()
    material1.material_id = 1
    material1.add_nuclide(nuclide, 1.0)
    material1.set_density("g/cm3", 1.0)
    material1.temperature = 294
    
    # Cell
    cell1 = mc.Cell(
        cell_id=1,
        name="inner_sphere",
        region=region1,
        fill=material1
    )
    geometry = mc.Geometry([cell1])
    
    # Get available MTs
    in_neu = mc.data.IncidentNeutron.from_hdf5(f"{NUCLEAR_DATA_DIR[nuclear_data]}neutron/{nuclide}.h5")
    mts_int = [mt for mt in in_neu.reactions.keys() if mt not in (1, 3, 4, 27, 101)]
    mts = [str(mt) for mt in mts_int]
    
    # Filters and tallies
    cell_filter = mc.CellFilter(cell1)
    energy_filter = mc.EnergyFilter.from_group_structure('VITAMIN-J-175')
    
    # Get energy bin boundaries - OpenMC returns (n_bins, 2) array with [low, high]
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])
    
    tallies = []
    
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:                                                                                                                                
        tally = mc.Tally()                                                                                                                                  
        tally.filters = [cell_filter]                                                                                                                       
        tally.scores = [score]                                                                                                                              
        tally.name = str(score)                                                                                                                             
        tallies.append(tally)    
    
    # Energy-binned flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)
    
    # Source and settings
    source = mc.IndependentSource(
        angle=mc.stats.Isotropic(),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        space=mc.stats.Point([0, 0, 0])
    )
    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False}
    )
    
    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)
    
    # Collect results
    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time
    
    results = {
        'nuclide': nuclide,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {}
    }
    
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]
    
    # Get spectrum tally
    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins
    
    return results

In [4]:
def run_yamc_simulation(nuclide, nuclear_data, compute="cpu"):
    """Run YAMC simulation for a single nuclide. Returns (results_dict, model)."""
    import yamc as mc

    # Geometry: broomstick cylinder
    cylinder = mc.Cylinder(axis='z', radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.Plane(axis='z', offset=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.Plane(axis='z', offset=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region1 = cylinder.below & z_bot.above & z_top.below

    material1 = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    # Keyword library - yamc autodownloads Arrow data into ~/.cache/yamc on first use
    material1.read_nuclear_data('endf-b8.1')

    mts = [mt for mt in material1.reaction_mts if mt not in (1, 3, 4, 27, 101)]

    cell1 = mc.Cell(
        id=1,
        name="inner_sphere",
        region=region1,
        material=material1,
    )
    geometry = mc.Geometry([cell1])

    tallies = []
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:
        tally = mc.Tally(
            scores=[str(score)],
            name=str(score),
            cells=cell1,
            particle="neutron",
        )
        tallies.append(tally)

    tally_spectrum = mc.Tally(
        scores=['flux'],
        name="flux_energy_binned",
        cells=cell1,
        particle="neutron",
        energy_group_structure='VITAMIN-J-175',
    )
    tallies.append(tally_spectrum)
    energy_bins = np.array(tally_spectrum.energy_bins)

    source = mc.NeutronSource(
        position=(0, 0, 0),
        direction=mc.sources.Isotropic(),
        energy=mc.sources.Discrete([SOURCE_ENERGY], [1.0]),
    )

    model = mc.Model(
        geometry=geometry,
        tallies=tallies,
        source=source,
        total_particles=PARTICLES * BATCHES,
        seed=1,
    )
    import time as _t
    _t0 = _t.perf_counter()
    sim_results = model.simulate_transport(compute=compute)
    _elapsed = _t.perf_counter() - _t0

    results = {
        'nuclide': nuclide,
        'code': f'yamc-{compute}',
        'particles/s': (
            sim_results.particles_per_second
            if sim_results.particles_per_second is not None
            else (PARTICLES * BATCHES) / _elapsed if _elapsed > 0 else 0
        ),
        'tallies': {},
        'tally_std': {}
    }
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            r = sim_results[tally]
            results['tallies'][tally.name] = r.mean[0]
            results['tally_std'][tally.name] = r.standard_deviation[0]

    spec = sim_results[tally_spectrum]
    results['flux_spectra_mean'] = np.array(spec.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(spec.standard_deviation).squeeze()
    results['energy_bins'] = energy_bins

    return results, model

In [5]:
def create_step_trace(energy_bins, values, name, color, dash=None, showlegend=True, legend='legend'):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
        showlegend=showlegend,
        legend=legend,
    )


# Per-backend plot styling. Keys are backend labels used in the legend.
BACKEND_STYLES = {
    'OpenMC CPU': dict(color='blue', dash=None, marker='circle'),
    'YAMC CPU': dict(color='red', dash='dash', marker='square'),
    'YAMC GPU': dict(color='orange', dash='dot', marker='diamond'),
}
RATIO_COLORS = {
    'YAMC CPU / OpenMC CPU': 'green',
    'YAMC GPU / OpenMC CPU': 'purple',
    'YAMC GPU / YAMC CPU': 'teal',
}
RATIO_FILLS = {
    'green': 'rgba(0, 128, 0, 0.2)',
    'purple': 'rgba(128, 0, 128, 0.2)',
    'teal': 'rgba(0, 128, 128, 0.2)',
}


def create_comparison_plot(openmc_results, yamc_results_by_backend):
    """Create a Plotly subplot with all spectra overlaid (top) and per-backend ratios (bottom).

    Parameters
    ----------
    openmc_results : dict or None
        OpenMC results dictionary. When None, the OpenMC trace is omitted
        and the ratio panel uses the first YAMC backend as reference. If
        only one YAMC backend is provided and openmc_results is None, the
        ratio panel is hidden.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    # Pick the reference for the ratio panel
    if openmc_results is not None:
        ref_label = 'OpenMC CPU'
        ratio_targets = [(f'YAMC {b.upper()}', res) for b, res in yamc_results_by_backend.items()]
    elif len(yamc_results_by_backend) >= 2:
        backends = list(yamc_results_by_backend)
        ref_label = f'YAMC {backends[0].upper()}'
        ratio_targets = [(f'YAMC {b.upper()}', yamc_results_by_backend[b]) for b in backends[1:]]
    else:
        ref_label = None
        ratio_targets = []

    # All spectra to show in the top panel
    spectra = []
    if openmc_results is not None:
        spectra.append(('OpenMC CPU', openmc_results))
    for backend, res in yamc_results_by_backend.items():
        spectra.append((f'YAMC {backend.upper()}', res))

    if not spectra:
        raise ValueError("create_comparison_plot needs at least one result")

    energy_bins = spectra[0][1]['energy_bins']
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])

    flux_by_label = {}
    std_by_label = {}
    for label, res in spectra:
        flux_by_label[label] = res['flux_spectra_mean'].copy() / lethargy_bin_width
        std_by_label[label] = res['flux_spectra_std_dev'].copy() / lethargy_bin_width

    # Plot range (lowest bin with nonzero flux in any spectrum)
    has_flux = np.zeros(len(energy_bins) - 1, dtype=bool)
    for f in flux_by_label.values():
        has_flux = has_flux | (f > 0)
    if np.any(has_flux):
        first_nonzero_idx = int(np.argmax(has_flux))
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    all_positive_max = []
    for f in flux_by_label.values():
        if np.any(f > 0):
            all_positive_max.append(np.max(f[f > 0]))
    max_flux = max(all_positive_max) if all_positive_max else 1e-10
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    def _mask(values, std, ref):
        keep = ref >= flux_threshold
        return np.where(keep, values, np.nan), np.where(keep, std, np.nan)

    show_ratio = bool(ratio_targets)
    if show_ratio:
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3],
            subplot_titles=('Flux per Unit Lethargy', f'Ratio (vs {ref_label})')
        )
    else:
        fig = make_subplots(
            rows=1, cols=1,
            subplot_titles=('Flux per Unit Lethargy',),
        )

    # Top panel: every spectrum
    for label, _res in spectra:
        f = flux_by_label[label]
        s = std_by_label[label]
        f_plot, s_plot = _mask(f, s, f)
        style = BACKEND_STYLES.get(label, dict(color='gray', dash=None, marker='circle'))
        fig.add_trace(
            create_step_trace(energy_bins, f_plot, label, style['color'], dash=style['dash']),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=bin_centers, y=f_plot, mode='markers',
                name=f'{label} error',
                marker=dict(color=style['color'], size=4, symbol=style['marker']),
                error_y=dict(type='data', array=s_plot, visible=True,
                             color=style['color'], thickness=1),
                showlegend=False,
            ),
            row=1, col=1
        )

    # Bottom panel: ratios against the reference
    if show_ratio:
        ref_flux = flux_by_label[ref_label]
        ref_std = std_by_label[ref_label]
        for label, _res in ratio_targets:
            f = flux_by_label[label]
            s = std_by_label[label]
            valid = (ref_flux > 0) & (f > 0)
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = np.where(valid, f / ref_flux, np.nan)
                rel_err_top = np.where(f > 0, s / f, 0)
                rel_err_ref = np.where(ref_flux > 0, ref_std / ref_flux, 0)
            ratio_rel_err = np.sqrt(rel_err_top**2 + rel_err_ref**2)
            ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

            ratio_label = f'{label} / {ref_label}'
            ratio_color = RATIO_COLORS.get(ratio_label, 'green')
            rgba_fill = RATIO_FILLS.get(ratio_color, 'rgba(0, 128, 0, 0.2)')

            fig.add_trace(
                create_step_trace(energy_bins, ratio, ratio_label, ratio_color, legend='legend2'),
                row=2, col=1
            )
        fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    # Axes
    fig.update_xaxes(
        type="log",
        title_text="Energy (eV)",
        range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
        exponentformat='e',
        row=(2 if show_ratio else 1), col=1
    )
    if show_ratio:
        fig.update_xaxes(
            type="log",
            range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
            exponentformat='e',
            row=1, col=1
        )
    fig.update_yaxes(type="log", title_text="Flux / Lethargy", exponentformat='e', row=1, col=1)
    if show_ratio:
        fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700 if show_ratio else 500,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top'),
        legend2=dict(x=0.02, y=0.27, xanchor='left', yanchor='top'),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(ref_results, other_results, flux_threshold=1e-6):
    """Statistical agreement metrics comparing one spectrum to a reference."""

    ref_flux = ref_results['flux_spectra_mean']
    ref_std = ref_results['flux_spectra_std_dev']
    oth_flux = other_results['flux_spectra_mean']
    oth_std = other_results['flux_spectra_std_dev']

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(ref_flux)):
        r = ref_flux[i]
        o = oth_flux[i]
        r_std = ref_std[i]
        o_std = oth_std[i]

        combined_std = np.sqrt(r_std**2 + o_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((o - r) / combined_std)**2)

        if r > flux_threshold or o > flux_threshold:
            if r > 0:
                rel_diff = abs((o - r) / r) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(r, o)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)

    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0

    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(ref_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# MT number to reaction description mapping from YAMC
import yamc
MT_DESCRIPTIONS = yamc.data.reaction_names()

# Score name to MT number mapping (reverse lookup + special score aliases)
SCORE_TO_MT = {name: mt for mt, name in MT_DESCRIPTIONS.items()}
SCORE_TO_MT.update({
    'total': 1,
    'elastic': 2,
    'absorption': 27,
    'fission': 18,
    'heating': 301,
    'heating-local': 901,
    'H1-production': 203,
    'H2-production': 204,
    'H3-production': 205,
    'He3-production': 206,
    'He4-production': 207,
})


def get_mt_and_reaction(score_name):
    """Get MT number and reaction description for a score.

    Returns (mt_display, reaction_description) tuple.
    """
    try:
        mt = int(score_name)
        return str(mt), MT_DESCRIPTIONS.get(mt, f'MT {mt}')
    except ValueError:
        pass

    if score_name in SCORE_TO_MT:
        mt = SCORE_TO_MT[score_name]
        return str(mt), score_name

    return '-', score_name


def _best_value(results, score_names):
    """Pick the best (non-nan, non-zero preferred) tally value across alias score names."""
    best_val = np.nan
    best_std = np.nan
    for name in score_names:
        val = results['tallies'].get(name, np.nan)
        std = results['tally_std'].get(name, np.nan)
        if not np.isnan(val) and val != 0:
            if np.isnan(best_val) or best_val == 0:
                best_val = val
                best_std = std
        elif np.isnan(best_val):
            best_val = val
            best_std = std
    return best_val, best_std


def _is_zero_or_nan(value):
    return value == 0 or np.isnan(value)


def _format_tally(value, std):
    if np.isnan(value):
        return 'N/A'
    return f"{value:.4e} +/- {std:.2e}"


def _format_rel_diff(value, ref_val):
    if np.isnan(ref_val) or ref_val == 0 or np.isnan(value):
        return 'N/A'
    return f"{(value - ref_val) / ref_val * 100:.2f}%"


def create_tally_table(openmc_results, yamc_results_by_backend):
    """Build a comparison table with one column per code/backend.

    Parameters
    ----------
    openmc_results : dict or None
        OpenMC results. When None, the OpenMC column is omitted and the
        first YAMC backend is used as the reference for relative differences.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    all_tallies = set()
    if openmc_results is not None:
        all_tallies |= set(openmc_results['tallies'].keys())
    for res in yamc_results_by_backend.values():
        all_tallies |= set(res['tallies'].keys())

    yamc_backends = list(yamc_results_by_backend)
    if openmc_results is not None:
        ref_results = openmc_results
        rel_diff_targets = yamc_backends
    elif yamc_backends:
        ref_results = yamc_results_by_backend[yamc_backends[0]]
        rel_diff_targets = yamc_backends[1:]
    else:
        return pd.DataFrame()

    mt_to_scores = {}
    for name in all_tallies:
        mt_display, _ = get_mt_and_reaction(name)
        mt_to_scores.setdefault(mt_display, []).append(name)

    rows = []
    for mt_display, score_names in mt_to_scores.items():
        if openmc_results is not None:
            openmc_val, openmc_std = _best_value(openmc_results, score_names)
        else:
            openmc_val, openmc_std = np.nan, np.nan
        yamc_vals = {b: _best_value(res, score_names) for b, res in yamc_results_by_backend.items()}
        ref_val, _ = _best_value(ref_results, score_names)

        all_zero = (
            (openmc_results is None or _is_zero_or_nan(openmc_val))
            and all(_is_zero_or_nan(v[0]) for v in yamc_vals.values())
        )
        if all_zero:
            continue

        reaction = None
        if mt_display != '-':
            try:
                reaction = MT_DESCRIPTIONS.get(int(mt_display))
            except ValueError:
                pass
        if reaction is None:
            _, reaction = get_mt_and_reaction(score_names[0])

        row = {'MT': mt_display, 'Reaction': reaction}
        if openmc_results is not None:
            row['OpenMC CPU'] = _format_tally(openmc_val, openmc_std)
        for backend, (val, std) in yamc_vals.items():
            row[f'YAMC {backend.upper()}'] = _format_tally(val, std)
        for backend in rel_diff_targets:
            val, _ = yamc_vals[backend]
            row[f'Rel. Diff {backend} (%)'] = _format_rel_diff(val, ref_val)

        row['_mt_sort'] = int(mt_display) if mt_display != '-' else 9999
        rows.append(row)

    rows.sort(key=lambda r: r['_mt_sort'])
    for row in rows:
        del row['_mt_sort']

    return pd.DataFrame(rows)

In [6]:
# Discover nuclides. When OpenMC runs, the HDF5 dir is the source of
# truth (we need OpenMC to have data too). When OpenMC is excluded, fall
# back to the local yamc Arrow cache so we aren't limited by partial
# OpenMC HDF5 downloads.
if RUN_OPENMC:
    h5_files = sorted(glob.glob(os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron', "*.h5")))
    nuclides = [os.path.splitext(os.path.basename(f))[0] for f in h5_files]
    if not nuclides:
        raise FileNotFoundError(
            f"No HDF5 files found in {NUCLEAR_DATA_DIR[NUCLEAR_DATA]}neutron/\n"
            f"Please check that NUCLEAR_DATA_DIR['{NUCLEAR_DATA}'] points to a valid nuclear data directory."
        )
else:
    import re
    from pathlib import Path as _P
    yamc_cache = _P.home() / ".cache" / "yamc"
    prefix = f"{NUCLEAR_DATA}-"
    nuclides = []
    for p in sorted(yamc_cache.glob(f"{prefix}*.arrow")):
        if not p.is_dir():
            continue
        name = p.name[len(prefix):-len(".arrow")]
        # Single-nuclide entries only (Ac225, Fe56, ...); drop
        # natural-element entries (Fe, U, ...) that have no mass number.
        if re.fullmatch(r"[A-Z][a-z]?\d+", name):
            nuclides.append(name)
    if not nuclides:
        raise FileNotFoundError(
            f"No yamc Arrow cache found at {yamc_cache} for library {NUCLEAR_DATA!r}.\n"
            f"Run yamc once on any nuclide to populate the cache."
        )

# Skip metastable nuclides for transport
nuclides = [n for n in nuclides if "_" not in n]

# Select which nuclide(s) to run. Honors (priority): the explicit
# YAMC_VERIFY_NUCLIDES list, then the YAMC_VERIFY_NUMBER_OF_SIMULATIONS
# count cap, else the full discovered list.
from cache_utils import select_requested
nuclides = select_requested(nuclides, kind="nuclide")

print(f"Running simulations for {len(nuclides)} nuclides: {nuclides}")

Running simulations for 1 nuclides: ['Fe56']


In [7]:
# Run all simulations and collect results. Each backend (openmc, yamc-cpu,
# yamc-gpu) runs in its own try/except, so a missing or failing backend never
# blocks the others -- any one can be built up independently.
all_results = {}
first_yamc_model = None

for nuclide in nuclides:
    print(f"\nProcessing {nuclide}...")
    per_nuclide = {}

    if RUN_OPENMC:
        try:
            per_nuclide['openmc'] = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "openmc", nuclide,
                lambda n=nuclide: run_openmc_simulation(n, NUCLEAR_DATA),
                meta=OPENMC_CACHE_META,
            )
            print(f"  OpenMC: {per_nuclide['openmc']['particles/s']:.0f} particles/s")
        except Exception as e:
            print(f"  OpenMC Error: {e}")

    for compute in COMPUTES:
        _captured = {}

        def _run_yamc(n=nuclide, c=compute, cap=_captured):
            res, model = run_yamc_simulation(n, NUCLEAR_DATA, compute=c)
            cap["model"] = model
            return res

        try:
            per_nuclide[f'yamc-{compute}'] = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, f"yamc-{compute}", nuclide, _run_yamc,
                meta=YAMC_CACHE_META,
            )
            if first_yamc_model is None and "model" in _captured:
                first_yamc_model = _captured["model"]
            print(f"  YAMC ({compute}): {per_nuclide[f'yamc-{compute}']['particles/s']:.0f} particles/s")
        except Exception as e:
            print(f"  YAMC ({compute}) Error: {e}")

    if per_nuclide:
        all_results[nuclide] = per_nuclide

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")


Processing Fe56...


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Simulating batch 2


 Simulating batch 3


 Simulating batch 4


 Simulating batch 5


... (14 outputs left out)

Progress: 45000000/50000000 particles (90%)  elapsed 16.7s, ETA ~1.9s

Progress: 50000000/50000000 particles (100%)  elapsed 18.5s, ETA ~0 ms
Transport simulation complete.
Time: 18.666 s (data load: 0.046 s, transport: 18.619 s) | Particles/s: 2,678,684 (transport only: 2,685,382) | Total: 50,000,000 particles
Tally statistics:
  2: mean=2.0234e-2 +/- 3.13e-6 (0.0% r.e.)  FOM=2.237e6  VOV=2.80e-5  checks=PASSED (7/7)
  5: mean=1.2461e-3 +/- 1.63e-7 (0.0% r.e.)  FOM=3.131e6  VOV=4.88e-6  checks=PASSED (7/7)
  16: mean=6.9659e-3 +/- 9.11e-7 (0.0% r.e.)  FOM=3.129e6  VOV=4.88e-6  checks=PASSED (7/7)
  51: mean=1.1413e-3 +/- 3.28e-7 (0.0% r.e.)  FOM=6.497e5  VOV=1.58e-4  checks=PASSED (7/7)
  52: mean=7.9576e-5 +/- 2.72e-8 (0.0% r.e.)  FOM=4.597e5  VOV=1.49e-4  checks=PASSED (7/7)
  53: mean=9.4904e-5 +/- 3.21e-8 (0.0% r.e.)  FOM=4.670e5  VOV=1.70e-4  checks=PASSED (7/7)
  54: mean=4.5243e-5 +/- 8.99e-9 (0.0% r.e.)  FOM=1.356e6  VOV=8.93e-5  checks=PASSED (7/7)
  55: mean=1.0042e-4 +/- 2.37e-8 (0.0% r.e.)  FOM=9.623e5  VOV=1.33e-4  checks=PASSED (7/7)
  56: 

GPU: Vulkan "AMD Radeon Graphics (RADV STRIX_HALO)" (IntegratedGpu, vendor 0x1002)


  yamc-gpu: cached to cache/endf-b8.1/yamc-gpu/Fe56.json
  YAMC (gpu): 1082674 particles/s

Completed 1 / 1 nuclides successfully.


In [8]:
# Display sphere model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("sphere_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

/home/jon/yamc-org/yamc-verification/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Interactive flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Tally comparison table sorted by relative difference
- Performance comparison (particles/second)

In [9]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_results = results.get('openmc')  # may be None when openmc-cpu not in MC_BACKENDS

    # Collect available YAMC backends in the order the user requested
    yamc_results_by_backend = {}
    for compute in COMPUTES:
        yres = results.get(f'yamc-{compute}')
        if yres is not None:
            yamc_results_by_backend[compute] = yres

    display(Markdown(f"## {nuclide}"))

    # Performance line - one entry per code/backend that ran
    perf_parts = []
    if openmc_results is not None:
        perf_parts.append(f"OpenMC: {openmc_results['particles/s']:,.0f} particles/s")
    for compute, yres in yamc_results_by_backend.items():
        perf_parts.append(f"YAMC ({compute}): {yres['particles/s']:,.0f} particles/s")
    display(Markdown(f"**Performance:** " + " | ".join(perf_parts)))

    # Combined flux spectra plot
    fig = create_comparison_plot(openmc_results, yamc_results_by_backend)
    fig.update_layout(title=f"{nuclide} - Flux Spectra Comparison")
    fig.show()

    # Spectral agreement: each yamc backend vs the reference (openmc when
    # present, else the first yamc backend).
    if openmc_results is not None:
        ref_label = 'OpenMC'
        ref_results = openmc_results
        comparisons = list(yamc_results_by_backend.items())
    elif len(yamc_results_by_backend) >= 2:
        backends = list(yamc_results_by_backend)
        ref_label = f'YAMC ({backends[0]})'
        ref_results = yamc_results_by_backend[backends[0]]
        comparisons = [(b, yamc_results_by_backend[b]) for b in backends[1:]]
    else:
        comparisons = []

    for compute, yres in comparisons:
        heading = (
            f"#### Spectral Agreement (YAMC {compute} vs {ref_label})"
            if len(comparisons) > 1 or openmc_results is None
            else "#### Spectral Agreement"
        )
        display(Markdown(heading))
        stats = calculate_spectral_agreement(ref_results, yres)
        display(create_agreement_table(stats).style.hide(axis='index'))

    # Combined tally table
    display(Markdown("#### Tally Comparison"))
    table = create_tally_table(openmc_results, yamc_results_by_backend)
    display(table.style.hide(axis='index'))

    print()  # spacing


## Fe56

**Performance:** OpenMC: 2,122,163 particles/s | YAMC (cpu): 2,678,684 particles/s | YAMC (gpu): 1,082,674 particles/s

#### Spectral Agreement (YAMC cpu vs OpenMC)

Metric,Value,Note
Reduced Chi-Squared,1.02,"(ideal ~ 1.0, dof=144)"
Weighted Avg Rel. Diff,0.05%,(flux-weighted)
Median Rel. Diff,2.14%,(robust to outliers)
Agreement Quality,Excellent,(128/175 bins analyzed)


#### Spectral Agreement (YAMC gpu vs OpenMC)

Metric,Value,Note
Reduced Chi-Squared,0.90,"(ideal ~ 1.0, dof=144)"
Weighted Avg Rel. Diff,0.04%,(flux-weighted)
Median Rel. Diff,1.84%,(robust to outliers)
Agreement Quality,Excellent,(126/175 bins analyzed)


#### Tally Comparison

MT,Reaction,OpenMC (cpu),YAMC (cpu),YAMC (gpu),Rel. Diff cpu (%),Rel. Diff gpu (%)
1,"(n,total)",4.3560e-02 +/- 7.76e-06,4.3556e-02 +/- 5.93e-06,4.3565e-02 +/- 5.78e-06,-0.01%,0.01%
2,"(n,elastic)",2.0235e-02 +/- 4.39e-06,2.0234e-02 +/- 3.13e-06,2.0238e-02 +/- 3.12e-06,-0.01%,0.02%
5,"(n,misc)",1.2463e-03 +/- 1.90e-07,1.2461e-03 +/- 1.63e-07,1.2464e-03 +/- 1.57e-07,-0.02%,0.01%
16,"(n,2n)",6.9670e-03 +/- 1.06e-06,6.9659e-03 +/- 9.11e-07,6.9674e-03 +/- 8.80e-07,-0.02%,0.01%
27,"(n,absorption)",1.9318e-03 +/- 2.95e-07,1.9315e-03 +/- 2.53e-07,1.9319e-03 +/- 2.45e-07,-0.02%,0.01%
51,"(n,n1)",1.1413e-03 +/- 4.54e-07,1.1413e-03 +/- 3.28e-07,1.1417e-03 +/- 3.27e-07,-0.00%,0.04%
52,"(n,n2)",7.9591e-05 +/- 3.00e-08,7.9576e-05 +/- 2.72e-08,7.9588e-05 +/- 2.61e-08,-0.02%,-0.00%
53,"(n,n3)",9.4932e-05 +/- 2.99e-08,9.4904e-05 +/- 3.21e-08,9.4910e-05 +/- 3.08e-08,-0.03%,-0.02%
54,"(n,n4)",4.5250e-05 +/- 7.83e-09,4.5243e-05 +/- 8.99e-09,4.5249e-05 +/- 8.56e-09,-0.02%,-0.00%
55,"(n,n5)",1.0043e-04 +/- 1.95e-08,1.0042e-04 +/- 2.37e-08,1.0043e-04 +/- 2.27e-08,-0.01%,-0.01%
